# Lab 03 · Pandas for Data Manipulation and Cleaning
**CS-3105 Machine Learning Lab** · Lab Instructor: Engr. Muhammad Awais

**CLO 6** (C-4 Analyze, PLO 2 Problem Analysis): *Analyze real-world datasets by applying data manipulation and visualization techniques.*

**Data:** `pakistan_used_cars.csv`, 24,973 real used-car ads from OLX Pakistan (Open Data Pakistan, *Pakistan Used Cars*, 2021). The ads are real and contain the problems real data has. Two kinds of formatting problem were added for practice (some prices written as text such as `Rs 1,430,000`, some city names in the wrong case or with extra spaces).

**Rules:** no Python loops over rows · label every output, e.g. `print("T4e:", ...)` · do not edit the CSV by hand · Runtime → Restart and run all before you submit · push this notebook to `ML-Lab-<RollNo>/Lab03/Lab03.ipynb` by **Thursday 5 November 2026, 11:59 PM (PKT)**.

Name: `SAQLAIN MUSHTAQ`  Roll No: `2024-SE-35`

### Setup
Put `pakistan_used_cars.csv` in the same folder as this notebook. In Colab: click the folder icon on the left → Upload → choose the CSV.

## Task 1 · Load and first look
- **(a)** Read `pakistan_used_cars.csv` into `df` and print its shape.
- **(b)** Show the first 5 rows.
- **(c)** Print the column names.
- **(d)** Show the data type of every column.
- **(e)** Which column should be a number but is not? Show two of its values that explain why.

In [1]:
import pandas as pd

df = pd.read_csv("pakistan_used_cars.csv")

print("T1a shape:", df.shape)
print("T1b first 5 rows:")
print(df.head())
print("T1c columns:", df.columns.tolist())
print("T1d data types:")
print(df.dtypes)

print("T1e Price is stored as:", df["Price"].dtype)
print("T1e two example values:", df.loc[df["Price"].astype(str).str.contains("Rs"), "Price"].head(2).tolist())

T1a shape: (24973, 9)
T1b first 5 rows:
    Brand Condition    Fuel  KMs Driven        Model    Price Registered City  \
0  Toyota      Used  Diesel         1.0        Prado  2100000         Karachi   
1  Suzuki      Used  Petrol    100000.0        Bolan   380000         Karachi   
2  Suzuki      Used     CNG     12345.0        Bolan   340000         Karachi   
3  Suzuki      Used  Petrol     94000.0         Alto   535000         Karachi   
4  Toyota      Used  Petrol    100000.0  Corolla XLI  1430000         Karachi   

  Transaction Type    Year  
0             Cash  1997.0  
1             Cash  2006.0  
2             Cash  1998.0  
3             Cash  2010.0  
4             Cash  2013.0  
T1c columns: ['Brand', 'Condition', 'Fuel', 'KMs Driven', 'Model', 'Price', 'Registered City', 'Transaction Type', 'Year']
T1d data types:
Brand                   str
Condition               str
Fuel                    str
KMs Driven          float64
Model                   str
Price               

**T1e:** `Price` should be numeric, but pandas reads it as text. Examples include values formatted as `Rs 435,000` and `Rs 4,150,000`; the currency prefix and commas prevent direct numeric calculations.

## Task 2 · Retrieve data
- **(a)** The `Brand` column (first 5 values).
- **(b)** The columns `Brand`, `Model`, `Price` together (first 5 rows).
- **(c)** The first row, using `iloc`.
- **(d)** The `Model` in the row with index 100, using `loc`.
- **(e)** How many cars are registered in `"Islamabad"`?
- **(f)** How many **Honda** cars run on **Hybrid** fuel?
- **(g)** Cars from 2018 or newer, newest first (Brand, Model, Year, Price; first 10).
- **(h)** Try `df[df["Price"] > 1_000_000]`. What happens and why? Answer in a Markdown cell.

In [2]:
print("T2a Brand (first 5):")
print(df["Brand"].head())

print("T2b Brand, Model and Price (first 5 rows):")
print(df.loc[:, ["Brand", "Model", "Price"]].head())

print("T2c first row using iloc:")
print(df.iloc[0])

print("T2d Model at index 100 using loc:", df.loc[100, "Model"])
print("T2e Islamabad registrations:", df[df["Registered City"] == "Islamabad"].shape[0])
print("T2f Honda Hybrid cars:", df[(df["Brand"] == "Honda") & (df["Fuel"] == "Hybrid")].shape[0])

print("T2g cars from 2018 onward, newest first (first 10):")
cars_2018 = df[df["Year"] >= 2018].sort_values("Year", ascending=False)
print(cars_2018.loc[:, ["Brand", "Model", "Year", "Price"]].head(10))

try:
    print(df[df["Price"] > 1_000_000])
except Exception as error:
    print(type(error).__name__ + ":", error)

T2a Brand (first 5):
0    Toyota
1    Suzuki
2    Suzuki
3    Suzuki
4    Toyota
Name: Brand, dtype: str
T2b Brand, Model and Price (first 5 rows):
    Brand        Model    Price
0  Toyota        Prado  2100000
1  Suzuki        Bolan   380000
2  Suzuki        Bolan   340000
3  Suzuki         Alto   535000
4  Toyota  Corolla XLI  1430000
T2c first row using iloc:
Brand                Toyota
Condition              Used
Fuel                 Diesel
KMs Driven              1.0
Model                 Prado
Price               2100000
Registered City     Karachi
Transaction Type       Cash
Year                 1997.0
Name: 0, dtype: object
T2d Model at index 100 using loc: Mehran VX
T2e Islamabad registrations: 958
T2f Honda Hybrid cars: 202
T2g cars from 2018 onward, newest first (first 10):
        Brand                      Model    Year    Price
10940  Toyota                     Probox  2020.0  1200000
13795   Honda                        Fit  2020.0  1950000
17124  Toyota                

The comparison attempts to compare text strings with a number. `Price` is stored as text because it may contain values such as `"Rs 1,430,000"`, so the filter raises an error (or produces unreliable/empty results in some pandas configurations). It must be cleaned and converted to a numeric type first.

## Task 3 · Clean the data: the six types of problem
Fix the problems **in this order** (text first, then duplicates):
- **(a) Missing values:** count them per column. Drop ads with no `Brand` or no `Model`; fill missing `Registered City`, `Fuel` and `Transaction Type` with `"Unknown"`; leave `Year` and `KMs Driven` as NaN. Print how many rows were removed.
- **(b) Formatting:** remove extra spaces and fix the case of city names (`.str.strip().str.title()`); remove `"Rs"` and commas from `Price`.
- **(c) Data types:** convert `Price` to `int64` and `Year` to `Int64`.
- **(d) Inconsistent categories:** `"Sukkar"` → `"Sukkur"` and `"Corrolla Altis"` → `"Corolla Altis"`. Print the number of different cities before and after.
- **(e) Duplicates:** count exact duplicate ads, drop them, print the new row count.
- **(f) Invalid values:** set `KMs Driven` to NaN when it is below 100, above 1,000,000, or the placeholder 12345 / 123456. Drop ads with the fake prices 12,345,678 or 87,654,321, and Suzuki/Daihatsu ads above 5,000,000.
- **(g) Self-check:** your final shape must be **(20484, 9)**.

In [3]:
# Missing values
print("T3a missing values per column:")
print(df.isna().sum())

rows_before = len(df)
df = df.dropna(subset=["Brand", "Model"])
print("T3a rows removed for missing Brand or Model:", rows_before - len(df))

for column in ["Registered City", "Fuel", "Transaction Type"]:
    df[column] = df[column].fillna("Unknown")

# Formatting
df["Registered City"] = df["Registered City"].str.strip().str.title()
df["Price"] = (df["Price"].astype(str)
               .str.replace("Rs", "", regex=False)
               .str.replace(",", "", regex=False)
               .str.strip())

# Data types
df["Price"] = pd.to_numeric(df["Price"]).astype("int64")
df["Year"] = df["Year"].astype("Int64")

# Inconsistent categories
cities_before = df["Registered City"].nunique()
df["Registered City"] = df["Registered City"].replace({"Sukkar": "Sukkur"})
df["Model"] = df["Model"].replace({"Corrolla Altis": "Corolla Altis"})
print("T3d different cities before corrections:", cities_before)
print("T3d different cities after corrections:", df["Registered City"].nunique())

# Duplicates
print("T3e exact duplicate ads:", df.duplicated().sum())
df = df.drop_duplicates()
print("T3e row count after dropping duplicates:", len(df))

# Invalid values
invalid_kms = ((df["KMs Driven"] < 100) |
               (df["KMs Driven"] > 1_000_000) |
               df["KMs Driven"].isin([12345, 123456]))
df.loc[invalid_kms, "KMs Driven"] = pd.NA

fake_prices = df["Price"].isin([12345678, 87654321])
df = df.loc[~fake_prices].copy()
df = df.loc[~(df["Brand"].isin(["Suzuki", "Daihatsu"]) & (df["Price"] > 5_000_000))].copy()

# Self-check
print("T3g final shape:", df.shape)
assert df.shape == (20484, 9)

T3a missing values per column:
Brand               2137
Condition           2136
Fuel                2445
KMs Driven          2286
Model               2448
Price                  0
Registered City     4636
Transaction Type    2445
Year                2284
dtype: int64
T3a rows removed for missing Brand or Model: 2448
T3d different cities before corrections: 62
T3d different cities after corrections: 61
T3e exact duplicate ads: 2028


T3e row count after dropping duplicates: 20497
T3g final shape: (20484, 9)


## Task 4 · Analyse
- **(a)** New column `Price_lakh` = price in lakh rupees (÷ 100,000).
- **(b)** New column `Age` = 2020 − `Year`.
- **(c)** The 5 most common brands.
- **(d)** The 5 cities with the most ads.
- **(e)** Median `Price_lakh` for each fuel type, highest first.
- **(f)** Median `Price_lakh` of a **Corolla GLI** for each year 2010–2018. How much value does it lose per year?
- **(g)** The 5 most expensive ads. Does one of them look wrong? Why?
- **(h)** Number of ads and median price for Cash vs Installment/Leasing. Interpret in 1–2 lines.

In [4]:
df["Price_lakh"] = df["Price"] / 100_000
df["Age"] = 2020 - df["Year"]

print("T4a Price_lakh and T4b Age created.")
print("T4c five most common brands:")
print(df["Brand"].value_counts().head(5))

print("T4d five cities with the most ads:")
print(df["Registered City"].value_counts().head(5))

print("T4e median Price_lakh by fuel type, highest first:")
print(df.groupby("Fuel", dropna=False)["Price_lakh"].median().sort_values(ascending=False))

gli = df[(df["Model"] == "Corolla GLI") &
         df["Year"].between(2010, 2018)]
gli_medians = gli.groupby("Year", observed=True)["Price_lakh"].median().sort_index()
print("T4f median Price_lakh for Corolla GLI, by year:")
print(gli_medians)

regression_data = gli_medians.dropna()
yearly_change = (regression_data.values[-1] - regression_data.values[0]) / (
    regression_data.index[-1] - regression_data.index[0])
print("T4f average increase per newer model year, in lakh:", round(yearly_change, 3))
print("T4f average value loss per model year of age, in lakh:", round(yearly_change, 3))

print("T4g five most expensive ads:")
print(df.nlargest(5, "Price")[["Brand", "Model", "Year", "Price", "Price_lakh", "Registered City"]])

transaction_summary = (
    df[df["Transaction Type"].isin(["Cash", "Installment/Leasing"])]
    .groupby("Transaction Type")["Price_lakh"]
    .agg(["count", "median"])
)
print("T4h ads and median price by transaction type:")
print(transaction_summary)

T4a Price_lakh and T4b Age created.
T4c five most common brands:
Brand
Suzuki      8621
Toyota      4535
Honda       3025
Daihatsu    2231
Nissan       694
Name: count, dtype: int64
T4d five cities with the most ads:
Registered City
Karachi       15461
Unknown        2040
Lahore         1601
Islamabad       777
Rawalpindi      119
Name: count, dtype: int64
T4e median Price_lakh by fuel type, highest first:
Fuel
Hybrid    16.2750
Diesel    13.0000
Petrol     8.7525
CNG        3.8500
LPG        1.5750
Name: Price_lakh, dtype: float64
T4f median Price_lakh for Corolla GLI, by year:
Year
2010    12.250
2011    13.325
2012    14.125
2013    14.950
2014    16.750
2015    17.900
2016    19.000
2017    21.000
2018    23.200
Name: Price_lakh, dtype: float64
T4f average increase per newer model year, in lakh: 1.369
T4f average value loss per model year of age, in lakh: 1.369
T4g five most expensive ads:
        Brand         Model  Year     Price  Price_lakh Registered City
19688  Toyota  Land C

                     count  median
Transaction Type                  
Cash                 19634     6.9
Installment/Leasing    850     3.0


The 670-lakh Toyota Land Cruiser price looks suspiciously high and may be a fake or mistyped listing. Cash listings also appear much more frequently and have a higher median price than Installment/Leasing listings. This is consistent with installment ads tending to feature more affordable cars and may also reflect the composition of the listings rather than a direct causal effect.

## Task 5 · Save your work
Save the cleaned DataFrame as `pakistan_used_cars_clean.csv` (Lab 04 uses it) and print its shape.

In [5]:
df.to_csv("pakistan_used_cars_clean.csv", index=False)
print("T5 cleaned data saved.")
print("T5 cleaned shape:", df.shape)

saved_df = pd.read_csv("pakistan_used_cars_clean.csv")
assert saved_df.shape == (20484, 11)
print("T5 verified saved shape:", saved_df.shape)

T5 cleaned data saved.
T5 cleaned shape: (20484, 11)


T5 verified saved shape: (20484, 11)


## Task 6 · Reflection (Markdown cell)
Write 3–5 sentences: which of the six cleaning problems would have given the most misleading answer if you had ignored it, and two things you learned about the used-car market in Pakistan.

Invalid values, fake prices, and duplicate records would have been especially misleading because they can distort totals and medians at the extremes. I learned that the Pakistani used-car market contains many Suzuki cars and that vehicle prices vary substantially by fuel type. The dataset also suggests newer Corolla GLI listings generally command higher prices and that cash listings are both more numerous and more expensive on average.